In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim 
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transform

transform = transform.Compose([
    transform.ToTensor(),
    transform.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [41:04<00:00, 69.2kB/s]
C:\Users\ASUS\anaconda3\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
train_loader = DataLoader(trainset , shuffle=True , batch_size=64)
test_loader = DataLoader(testset , batch_size=64)

### CNN Architechture

In [13]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),                      #(32,32,32)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2),    # Kernel_size=2 , stride_val=2          #(16,16,32)    
                                                                          #     |
            nn.Conv2d(32,64,kernel_size=3,padding=1),                     #(16,16,64)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2),                                            # (8,8,64) 
                                                                          #     |
            nn.Conv2d(64,128,kernel_size=3,padding=1),                    # (8,8,128)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2)    # Kernel_size=2 , stride_val=2           # (4,4,128)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128 , 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

        def forward(self,x):
            x = self.conv_layer(x)
            x = x.view(x.size(0),-1)  # Flattening 
            x = self.fc_layer(x)

            return x

In [14]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())